# Day 04 미니 프로젝트 · 사내 지식 리소스 서버
**Day 01~02에서 다룬 그 사내 문서**(연차·비용·보안·제품…)를 이제 **MCP Resource로 노출**합니다.
읽기 전용 리소스로 본문·목록·카테고리를 제공해, 에이전트가 "무엇을 읽을 수 있는지" 발견하고 가져가게 만듭니다.
핵심 리소스는 **빈칸(TODO)** — 실습(Day04)을 참고해 채우세요.

### 진행
- Part 1 · 사내 문서 + 서버 (주어짐)
- Part 2 · 정적 리소스 `info://about` (주어짐, 워밍업)
- Part 3 · 문서 본문 `doc://{doc_id}` (빈칸) — 실습 Lab 2
- Part 4 · 전체 목록 `index://docs` (빈칸) — 실습 Lab 3
- Part 5 · 카테고리 `category://{cat}` (빈칸) — **Day02 메타데이터 필터를 리소스로**
- Part 6 · Client 시나리오 · Part 7 · 자동 채점
---

## 0 · 설치

In [ ]:
%pip install -q fastmcp

In [ ]:
import fastmcp
from fastmcp import FastMCP, Client
print("fastmcp", fastmcp.__version__)

## Part 1 · 사내 문서 + 서버
Day02의 도메인 그대로입니다. 각 문서에 `doc_id·title·category` 메타데이터가 붙어 있습니다 (Day02의 그 메타데이터).

In [ ]:
DOCS = [
    {"doc_id": "leave",    "title": "연차 및 휴가 규정", "category": "인사", "body": "연차 휴가는 사용 3영업일 전까지 사내포털에서 신청하고 팀장 승인을 받는다."},
    {"doc_id": "expense",  "title": "비용 정산 절차",   "category": "재무", "body": "비용 정산은 영수증을 첨부해 정산 메뉴에서 접수하고 재무팀이 최종 승인한다."},
    {"doc_id": "security", "title": "정보보안 정책",     "category": "보안", "body": "외부 USB는 원칙적으로 금지되며 파일 공유는 사내 승인 드라이브만 사용한다."},
    {"doc_id": "remote",   "title": "재택근무 및 근태",  "category": "인사", "body": "재택근무는 주 2회까지 가능하며 전날 18시까지 팀 채널에 공유한다."},
    {"doc_id": "err4021",  "title": "오류 코드 안내",     "category": "IT",   "body": "에러코드 ERR-4021은 인증 토큰 만료를 의미하며 재로그인으로 해결한다."},
    {"doc_id": "x100",     "title": "제품 X-100 매뉴얼",  "category": "제품", "body": "제품 X-100의 펌웨어 최신 버전은 3.0이며 전용 앱의 설정 메뉴에서 업데이트한다."},
]
mcp = FastMCP("KnowledgeServer")
print("문서", len(DOCS), "건 | 카테고리:", sorted({d["category"] for d in DOCS}))

## Part 2 · 정적 리소스 `info://about` (주어짐)
고정 URI + description. 이 서버가 무엇인지 알립니다 (실습 Lab 1).

In [ ]:
@mcp.resource("info://about", description="이 서버 소개")
def about() -> str:
    return "Day01~02의 사내 문서를 읽기 전용으로 제공하는 지식 서버"

print("info://about 등록")

## Part 3 · 문서 본문 `doc://{doc_id}` (빈칸)
`{doc_id}`가 함수 인자로 들어옵니다 (실습 Lab 2의 `policy://{topic}` 참고).

In [ ]:
# TODO(실습 Lab 2 참고): @mcp.resource("doc://{doc_id}", description=...) 로
#   doc_id에 해당하는 문서의 body를 반환하세요. 없으면 "해당 문서가 없습니다."

# 여기에 구현

print("doc://{doc_id} 등록")

## Part 4 · 전체 목록 `index://docs` (빈칸)
무엇을 읽을 수 있는지 알려 발견 가능성을 높입니다 (실습 Lab 3).

In [ ]:
# TODO(실습 Lab 3 참고): @mcp.resource("index://docs", description=...) 로
#   각 문서의 {doc_id, title, category} 목록을 반환하세요.

# 여기에 구현

print("index://docs 등록")

## Part 5 · 카테고리 `category://{cat}` (빈칸)
**Day02의 메타데이터 필터를 그대로 리소스로** 옮깁니다. 카테고리로 문서를 좁혀 반환합니다.

In [ ]:
# TODO: @mcp.resource("category://{cat}", description=...) 로
#   category가 cat인 문서들을 반환하세요. (Day02 메타데이터 필터와 같은 아이디어)
#   팁: 문자열만 든 list 대신 dict로 감싸면 안전 — 예: {"category": cat, "docs": [...]}

# 여기에 구현

print("category://{cat} 등록")

## Part 6 · Client 시나리오
in-process Client로 붙어, 소개 → 목록 → 본문 → 카테고리 순으로 읽어봅니다.

In [ ]:
async with Client(mcp) as client:
    print("소개  :", (await client.read_resource("info://about"))[0].text)
    print("목록  :", (await client.read_resource("index://docs"))[0].text[:80], "...")
    print("본문  :", (await client.read_resource("doc://leave"))[0].text)
    print("인사 카테고리:", (await client.read_resource("category://인사"))[0].text)

## Part 7 · 자동 채점 — 기능 점수
빈칸을 제대로 채웠는지 4개 항목으로 채점합니다. 4/4가 목표입니다.

In [ ]:
async def grade():
    score, log = 0, []
    async with Client(mcp) as c:
        checks = [
            ("doc://leave 본문",  "doc://leave",    lambda t: "연차" in t),
            ("index://docs 목록", "index://docs",   lambda t: "leave" in t and "x100" in t),
            ("category://인사",   "category://인사", lambda t: "remote" in t and "leave" in t),
            ("info://about",      "info://about",   lambda t: len(t) > 0),
        ]
        for name, uri, cond in checks:
            try:
                t = (await c.read_resource(uri))[0].text; ok = bool(cond(t))
            except Exception as e:
                ok = False
            score += ok; log.append(f"[{'O' if ok else 'X'}] {name}")
    print("\n".join(log)); print(f"\n기능 점수: {score}/4")

await grade()

## Part 8 · 도전 & 체크리스트 (→ Day05 예고)
- **A** `doc://{doc_id}`가 없는 id면 친절한 에러를 던지도록(입력 검증) — Day05 예고
- **B** 청크를 `chunk://{doc_id}` 로 노출해 검색용 조각 단위 제공
- **C** 조회는 Resource, 수정은 Tool로 나눠보기 — **다음 시간(Day05)** 에 이 문서 서버에 검색·행동 도구를 붙여 실무형 서버로 완성합니다.

체크리스트
- [ ] 정적/동적/목록/카테고리 리소스를 모두 등록했다
- [ ] 카테고리 필터(Day02 메타데이터)를 리소스로 옮겼다
- [ ] Client 시나리오와 자동 채점(4/4)을 통과했다